# IEEE-CIS MLP-PLR Fusion Benchmark on Colab

Notebook nay dung de benchmark `MLP-PLR score fusion` voi backbone tabular hien tai tren Google Colab.

## Runtime

Trong Colab, dat `Runtime -> Change runtime type -> Hardware accelerator -> GPU` truoc khi chay.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
GITHUB_REPO = 'https://github.com/Thanh-000/FDB1-DoAn.git'
REPO_PARENT = '/content/drive/MyDrive'
REPO_NAME = 'FDB1-DoAn'
REPO_DIR = f'{REPO_PARENT}/{REPO_NAME}'

ZIP_PATH = '/content/drive/MyDrive/MVS_XAI_Data/ieee-fraud-detection.zip'
LOCAL_ZIP_PATH = '/content/ieee-fraud-detection.zip'
EXTRACT_DIR = '/content/ieee-fraud-detection'

FOLD_INDEX = 0
N_SPLITS = 5
INNER_SPLITS = 2
EPOCHS = 10
BATCH_SIZE = 2048
HIDDEN_DIM = 256
N_BLOCKS = 4
N_FREQUENCIES = 8
USE_TIME_FEATURE = False
LR = 1e-3


In [ ]:
import os
import subprocess

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', GITHUB_REPO, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

print('REPO_DIR =', REPO_DIR)


In [ ]:
import glob
import os
import shutil
import zipfile

if not os.path.exists(LOCAL_ZIP_PATH):
    shutil.copy2(ZIP_PATH, LOCAL_ZIP_PATH)

if not os.path.exists(EXTRACT_DIR):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(LOCAL_ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)

txn_files = glob.glob(f'{EXTRACT_DIR}/**/train_transaction.csv', recursive=True)
id_files = glob.glob(f'{EXTRACT_DIR}/**/train_identity.csv', recursive=True)
if not txn_files or not id_files:
    raise FileNotFoundError('Could not locate train_transaction.csv/train_identity.csv after extraction')
DATA_DIR = os.path.dirname(txn_files[0])
print('DATA_DIR =', DATA_DIR)


In [ ]:
%cd "$REPO_DIR"
!pwd
!ls
!ls mlp_plr


In [ ]:
!pip -q install scikit-learn torch xgboost lightgbm catboost


In [ ]:
import sys
import torch

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO CUDA')


In [ ]:
import shlex
import subprocess

cmd = [
    'python', 'run_ieee_mlp_plr_fusion.py',
    '--data-dir', DATA_DIR,
    '--fold-index', str(FOLD_INDEX),
    '--n-splits', str(N_SPLITS),
    '--inner-splits', str(INNER_SPLITS),
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--hidden-dim', str(HIDDEN_DIM),
    '--n-blocks', str(N_BLOCKS),
    '--n-frequencies', str(N_FREQUENCIES),
    '--lr', str(LR),
]

if USE_TIME_FEATURE:
    cmd.append('--use-time-feature')

print('MLP-PLR-FUSION config:')
print('  FOLD_INDEX =', FOLD_INDEX)
print('  INNER_SPLITS =', INNER_SPLITS)
print('  N_FREQUENCIES =', N_FREQUENCIES)
print('  USE_TIME_FEATURE =', USE_TIME_FEATURE)
print('Running:')
print(' '.join(shlex.quote(x) for x in cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
returncode = proc.wait()
print('returncode =', returncode)
if returncode != 0:
    raise subprocess.CalledProcessError(returncode, cmd)
